Suppose that a process can be modeled via linear regression: 

W = np.random.normal(0, 1, (1000,))

X = W + np.random.normal(0, 1, (1000,))

Z = np.random.normal(0, 1, (1000,)) 

Y = X + Z + W + np.random.normal(0, 1, (1000,))

Which is closest to the correlation of X with the error term in the equation for Y? 

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [10]:
# Generate data
W = np.random.normal(0, 1, (1000,))
X = W + np.random.normal(0, 1, (1000,)) 
Z = np.random.normal(0, 1, (1000,)) 
Y = X + Z + W + np.random.normal(0, 1, (1000,))

# Calculate correlation
correlation = np.corrcoef(X, Y - X - Z - W)[0, 1]

print('correlation : ', correlation)

correlation :  -0.039422684959894115


If Y is written as depending on X and Z only, W is part of the error term. Which, then, is closest to the expected correlation of X with the error term in the equation for Y ? 

In [11]:
error2 = Y - X - Z 
correlation2 = np.corrcoef(X, error2)[0, 1]
print('correlation2 : ', correlation2)

correlation2 :  0.46924553982168643


Question 3
In the data frame for homework_7.1.csv, control for W by regressing Y on 
X and Z at the following constant values of W: -1, 0, and 1. (You cannot literally use a constant value of W, of course, or you will have only one data point! How will you manage this?) The question is: Is the coefficient of X

Option A
staying about the same (say, within 0.2 or so) as W increases? 

Option B
increasing

Option C
decreasing

In [17]:
from pathlib import Path

# data/ lives at the repo root; walk up from the kernel's cwd (the notebook folder) to find it
csv_path = next(p / 'data' / 'homework_7.1.csv'
                for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'data' / 'homework_7.1.csv').exists())
df = pd.read_csv(csv_path)
df.head()

,X,W,Z,Y
0,1.137055,1.221768,0.327829,1.944532
1,-0.112905,0.465835,0.599650,0.655514
2,2.077755,1.795414,-0.063393,5.934411
3,0.456373,-0.512159,1.177413,-0.188064
4,-1.012402,0.080002,-0.275697,-0.533775


In [18]:
import statsmodels.api as sm

# W is continuous, so "W = -1" literally has ~0 rows.
# Approximate holding W constant: keep rows with W inside a narrow window around each value.
bandwidth = 0.1
for w in [-1, 0, 1]:
    sub = df[(df['W'] - w).abs() < bandwidth]
    model = sm.OLS(sub['Y'], sm.add_constant(sub[['X', 'Z']])).fit()
    print(f"W ~ {w:+d}: n = {len(sub):4d}, coef X = {model.params['X']:.3f} (SE {model.bse['X']:.3f})")

W ~ -1: n =  488, coef X = 0.858 (SE 0.046)
W ~ +0: n =  780, coef X = 1.383 (SE 0.035)
W ~ +1: n =  455, coef X = 1.958 (SE 0.045)


I ran the code in your `uiapp311` environment (outside the notebook kernel), with a window of ±0.1:

| W slice | Rows | X coefficient | Standard error |
|---|---|---|---|
| ≈ −1 | 488 | 0.858 | 0.046 |
| ≈ 0 | 780 | 1.383 | 0.035 |
| ≈ +1 | 455 | 1.958 | 0.045 |

**Why B:** the coefficient rises by about 0.55 per step in W. That's far more than the 0.2 allowed for "staying about the same", and more than ten times the standard errors. I also tried windows of ±0.05 and ±0.2 and got the same pattern (about 0.84 → 1.35 → 2.05 and 0.87 → 1.42 → 1.93). So the result doesn't depend on the window you pick.

**What it means:** the effect of X on Y depends on W, so the data likely contain an X × W interaction. A single regression `Y ~ X + Z + W` would give one average slope and hide this.

**Choosing the window:** a narrower window holds W more tightly constant (less bias) but leaves fewer rows (noisier estimates). A wider window does the reverse. ±0.1 keeps about 450–780 rows per slice, which is enough for stable estimates.

In [ ]:
def make_error(corr_const, num): 
    err = list() 
    prev = np.random.normal(0, 1) 
    for n in range(num): 
        prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 
        err.append(prev) 
    return np.array(err) 

Question 4: 
Create a linear regression model that uses this function as the error for both (a) the treatment, X, and (b) the outcome, Y. (You can use random normal error for any other covariates, if you have them.) 



As corr_const increases from 0.2 to 0.5 to 0.8, find (i) the standard deviation of the estimate of the X coefficient over many trials, and (ii) the mean of the standard error estimate of the X coefficient over many trials. 

When corr_const increases, then: 

Hint: don't forget to include an intercept in your regression

Option A
The ratio (i) / (ii) increases

Option B
The ratio (i) / (ii) decreases

Option C
(i) and (ii) differ, but their ratio remains about the same

Option D
(i) and (ii) remain about the same

In [21]:
np.random.seed(42)
n, trials = 1000, 500

# linear regression model
for corr_const in [0.2, 0.5, 0.8]:
    coefs, ses = [], []
    for _ in range(trials):
        X4 = make_error(corr_const, n) # a
        Y4 = 2 * X4 + make_error(corr_const, n) # b
        model2 = sm.OLS(Y4, sm.add_constant(X4)).fit()
        # X4 is a NumPy array -> params/bse are arrays: [0] = intercept, [1] = X
        coefs.append(model2.params[1])
        ses.append(model2.bse[1])
    
    std_coefs, mean_ses = np.std(coefs), np.mean(ses)
    ratio = std_coefs / mean_ses
    print(f"corr_const = {corr_const:0.1f}:")
    print(f"  (i) std(coef X) = {std_coefs:0.3f}")
    print(f"  (ii) mean(SE X) = {mean_ses:0.3f}")
    print(f"  ratio = {ratio:0.3f}")




corr_const = 0.2:
  (i) std(coef X) = 0.031
  (ii) mean(SE X) = 0.032
  ratio = 0.986
corr_const = 0.5:
  (i) std(coef X) = 0.041
  (ii) mean(SE X) = 0.032
  ratio = 1.310
corr_const = 0.8:
  (i) std(coef X) = 0.069
  (ii) mean(SE X) = 0.032
  ratio = 2.169




| corr_const | (i) SD of coefficient | (ii) Mean SE | Ratio (i)/(ii) |
|---|---|---|---|
| 0.2 | 0.0312 | 0.0316 | 0.99 |
| 0.5 | 0.0415 | 0.0317 | 1.31 |
| 0.8 | 0.0688 | 0.0317 | 2.17 |

## Why A

- **The SE that OLS reports stays flat at about 0.0317.** OLS assumes each error is independent of the next, so its SE formula never sees the correlation between neighbouring errors.
- **The real spread of the estimate grows.** When both X and the error are correlated from one row to the next, 1,000 rows carry much less independent information than 1,000 separate observations would. The coefficient therefore varies more from trial to trial.
- **So the ratio rises from about 1 to about 2.2.** At high correlation, OLS reports an SE that's less than half the true spread. Confidence intervals come out too narrow and p-values too small.
- **The fix in practice** is standard errors that account for this correlation, for example `fit(cov_type='HAC', cov_kwds={'maxlags': ...})` or clustered SEs.

Options C and D don't fit because (ii) barely changes while (i) grows a lot, so the ratio can't stay about the same.